In [1]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Change runtime to GPU before continuing.")

Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
# Cell 02: Install and import Torch-Pruning
# This cell installs Torch-Pruning, which we will use for structured/channel pruning.

!pip install torch-pruning -q

import torch_pruning as tp

print("Torch-Pruning imported successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 7.2 MB/s eta 0:00:00
Torch-Pruning imported successfully


In [3]:
# Cell 03: Mount Google Drive
# This cell connects Google Drive so that Colab can access the thesis project folder, dataset CSV files, and baseline model.

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Cell 04: Set project paths
# This cell defines all important project paths for dataset, baseline model, optimized model folder, and result folder.

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Thesis/thesis_project")

DATA_DIR = PROJECT_DIR / "data"
BASELINE_MODEL_PATH = PROJECT_DIR / "01_baseline_before_optimization/models/mobilenetv2_baseline_best.pth"

OPT_DIR = PROJECT_DIR / "02_after_pruning_quantization"
NOTEBOOK_DIR = OPT_DIR / "notebooks"
MODEL_DIR = OPT_DIR / "models"
RESULTS_DIR = OPT_DIR / "results"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project exists:", PROJECT_DIR.exists())
print("Data exists:", DATA_DIR.exists())
print("Baseline model exists:", BASELINE_MODEL_PATH.exists())

print("\nProject directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Baseline model path:", BASELINE_MODEL_PATH)
print("Model output directory:", MODEL_DIR)
print("Results output directory:", RESULTS_DIR)

Project exists: True
Data exists: True
Baseline model exists: True

Project directory: /content/drive/MyDrive/Thesis/thesis_project
Data directory: /content/drive/MyDrive/Thesis/thesis_project/data
Baseline model path: /content/drive/MyDrive/Thesis/thesis_project/01_baseline_before_optimization/models/mobilenetv2_baseline_best.pth
Model output directory: /content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/models
Results output directory: /content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/results


In [5]:
# Cell 05: Import libraries and set configuration
# This cell imports required libraries and sets seed, image size, batch size, pruning ratios, and fine-tuning settings.

import os
import time
import copy
import random
import json

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm.auto import tqdm

import torch_pruning as tp

SEED = 42
IMG_SIZE = 224
NUM_CLASSES = 38
BATCH_SIZE = 32

# Candidate pruning ratios for maximum measurable output
PRUNING_RATIOS = [0.10, 0.20, 0.30]

# Short fine-tuning first; later we can fine-tune the best candidate more if needed
EPOCHS_PER_CANDIDATE = 3
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("Pruning ratios:", PRUNING_RATIOS)
print("Epochs per candidate:", EPOCHS_PER_CANDIDATE)

Device: cuda
GPU: Tesla T4
Pruning ratios: [0.1, 0.2, 0.3]
Epochs per candidate: 3


In [21]:
# Cell 06: Copy PlantVillage zip from Google Drive, unzip it in Colab, and detect the correct dataset root.
# This cell prepares the actual image dataset for pruning and fine-tuning.

from pathlib import Path
import shutil
import zipfile
import pandas as pd

# Dataset zip found in your Google Drive
DRIVE_ZIP_PATH = Path("/content/drive/MyDrive/Thesis/Dataset/PlantVillage.zip")

# Local Colab paths
LOCAL_ZIP_PATH = Path("/content/PlantVillage.zip")
EXTRACT_DIR = Path("/content/PlantVillage_extracted")

# CSV paths
train_csv = DATA_DIR / "train_files.csv"
val_csv = DATA_DIR / "val_files.csv"

print("Drive zip exists:", DRIVE_ZIP_PATH.exists())
print("Drive zip path:", DRIVE_ZIP_PATH)

if not DRIVE_ZIP_PATH.exists():
    raise FileNotFoundError("PlantVillage.zip was not found. Check the Google Drive path.")

# Copy zip to Colab local storage for faster extraction
if (not LOCAL_ZIP_PATH.exists()) or (LOCAL_ZIP_PATH.stat().st_size != DRIVE_ZIP_PATH.stat().st_size):
    print("\nCopying PlantVillage.zip from Google Drive to Colab local storage...")
    shutil.copy2(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)
    print("Copy completed.")
else:
    print("\nLocal PlantVillage.zip already exists. Skipping copy.")

print("Local zip exists:", LOCAL_ZIP_PATH.exists())
print("Local zip path:", LOCAL_ZIP_PATH)

# Read first image path from train CSV
train_df_check = pd.read_csv(train_csv)
first_relative_path = train_df_check.iloc[0]["relative_path"]

print("\nFirst relative path from CSV:")
print(first_relative_path)

# Function to detect dataset root
def detect_dataset_root(base_dir, relative_path):
    candidate_roots = [
        base_dir,
        base_dir / "PlantVillage",
        base_dir / "PlantVillage" / "PlantVillage",
        base_dir / "PlantVillage" / "color",
        base_dir / "PlantVillage" / "Color",
        base_dir / "color",
        base_dir / "Color",
        base_dir / "raw",
        base_dir / "Raw",
    ]

    for root in candidate_roots:
        test_path = root / relative_path
        print("Checking root:", root)
        print("Test image exists:", test_path.exists())
        print("-" * 80)

        if test_path.exists():
            return root

    # Recursive fallback search for the class folder
    class_folder = Path(relative_path).parts[0]
    print("\nTrying recursive search for class folder:", class_folder)

    for found_class_dir in base_dir.rglob(class_folder):
        possible_root = found_class_dir.parent
        test_path = possible_root / relative_path

        print("Recursive possible root:", possible_root)
        print("Test image exists:", test_path.exists())
        print("-" * 80)

        if test_path.exists():
            return possible_root

    return None

# Create extraction folder
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# First try detecting root if already extracted
print("\nTrying to detect dataset root before extraction...")
DETECTED_DATASET_ROOT = detect_dataset_root(EXTRACT_DIR, first_relative_path)

# If not found, extract zip
if DETECTED_DATASET_ROOT is None:
    print("\nDataset root not found yet. Extracting zip now...")
    with zipfile.ZipFile(LOCAL_ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("Extraction completed.")

    print("\nTrying to detect dataset root after extraction...")
    DETECTED_DATASET_ROOT = detect_dataset_root(EXTRACT_DIR, first_relative_path)

if DETECTED_DATASET_ROOT is None:
    print("\nCould not detect dataset root.")
    print("Extraction folder contents:")
    for item in EXTRACT_DIR.iterdir():
        print("-", item.name)
    raise FileNotFoundError("Dataset root could not be detected. Send me the printed folder list.")

print("\nFinal detected dataset root:")
print(DETECTED_DATASET_ROOT)

Drive zip exists: True
Drive zip path: /content/drive/MyDrive/Thesis/Dataset/PlantVillage.zip

Copying PlantVillage.zip from Google Drive to Colab local storage...
Copy completed.
Local zip exists: True
Local zip path: /content/PlantVillage.zip

First relative path from CSV:
Tomato___Tomato_Yellow_Leaf_Curl_Virus/0fada544-d797-4168-bbf4-4f9dcfd24ec8___YLCV_NREC 1993.JPG

Trying to detect dataset root before extraction...
Checking root: /content/PlantVillage_extracted
Test image exists: False
--------------------------------------------------------------------------------
Checking root: /content/PlantVillage_extracted/PlantVillage
Test image exists: False
--------------------------------------------------------------------------------
Checking root: /content/PlantVillage_extracted/PlantVillage/PlantVillage
Test image exists: False
--------------------------------------------------------------------------------
Checking root: /content/PlantVillage_extracted/PlantVillage/color
Test image 

In [22]:
# Cell 07: Create dataset class, transforms, and dataloaders.
# This cell loads train and validation images using relative_path and label columns.

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import pandas as pd

DATASET_ROOT = Path(DETECTED_DATASET_ROOT)

class PlantVillageCSVDataset(Dataset):
    def __init__(self, csv_path, dataset_root_dir, transform=None):
        self.csv_path = Path(csv_path)
        self.dataset_root_dir = Path(dataset_root_dir)
        self.transform = transform

        self.df = pd.read_csv(self.csv_path)

        self.path_col = "relative_path"
        self.label_col = "label"

        print(f"Loaded CSV: {self.csv_path}")
        print("Rows:", len(self.df))
        print("Columns:", self.df.columns.tolist())
        print("Dataset root:", self.dataset_root_dir)

        if self.path_col not in self.df.columns:
            raise ValueError(f"CSV must contain '{self.path_col}' column")

        if self.label_col not in self.df.columns:
            raise ValueError(f"CSV must contain '{self.label_col}' column")

        sample_path = self.dataset_root_dir / self.df.iloc[0][self.path_col]
        print("Sample image path:", sample_path)
        print("Sample image exists:", sample_path.exists())

        if not sample_path.exists():
            raise FileNotFoundError("Sample image not found. DATASET_ROOT is incorrect.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = self.dataset_root_dir / row[self.path_col]
        label = int(row[self.label_col])

        # Convert every image to RGB because one image in your dataset is RGBA.
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = PlantVillageCSVDataset(
    csv_path=train_csv,
    dataset_root_dir=DATASET_ROOT,
    transform=train_transform
)

val_dataset = PlantVillageCSVDataset(
    csv_path=val_csv,
    dataset_root_dir=DATASET_ROOT,
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("\nFinal dataset check:")
print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Dataset root:", DATASET_ROOT)

Loaded CSV: /content/drive/MyDrive/Thesis/thesis_project/data/train_files.csv
Rows: 40728
Columns: ['relative_path', 'class_name', 'label']
Dataset root: /content/PlantVillage_extracted/PlantVillage
Sample image path: /content/PlantVillage_extracted/PlantVillage/Tomato___Tomato_Yellow_Leaf_Curl_Virus/0fada544-d797-4168-bbf4-4f9dcfd24ec8___YLCV_NREC 1993.JPG
Sample image exists: True
Loaded CSV: /content/drive/MyDrive/Thesis/thesis_project/data/val_files.csv
Rows: 8146
Columns: ['relative_path', 'class_name', 'label']
Dataset root: /content/PlantVillage_extracted/PlantVillage
Sample image path: /content/PlantVillage_extracted/PlantVillage/Grape___Black_rot/5ce9d6df-162c-4488-9b80-eadfcbd1d02c___FAM_B.Rot 0563.JPG
Sample image exists: True

Final dataset check:
Train samples: 40728
Validation samples: 8146
Dataset root: /content/PlantVillage_extracted/PlantVillage


In [23]:
# Cell 08: Create MobileNetV2 and load the best baseline checkpoint.
# This cell rebuilds MobileNetV2 and loads the saved baseline model from epoch 9.

import torch
import torch.nn as nn
from torchvision import models

def create_mobilenetv2(num_classes=38):
    model = models.mobilenet_v2(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model


def load_baseline_model():
    model = create_mobilenetv2(NUM_CLASSES)

    checkpoint = torch.load(
        BASELINE_MODEL_PATH,
        map_location=device,
        weights_only=False
    )

    print("Checkpoint keys:")
    if isinstance(checkpoint, dict):
        for key in checkpoint.keys():
            print("-", key)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])

        print("\nLoaded checkpoint['model_state_dict']")
        print("Checkpoint epoch:", checkpoint.get("epoch", "Not found"))
        print("Best validation accuracy:", checkpoint.get("best_val_accuracy", "Not found"))
        print("Number of classes:", checkpoint.get("num_classes", "Not found"))
        print("Image size:", checkpoint.get("image_size", "Not found"))
        print("Batch size:", checkpoint.get("batch_size", "Not found"))
        print("Model name:", checkpoint.get("model_name", "Not found"))
    else:
        model.load_state_dict(checkpoint)
        print("Loaded direct state_dict")

    model = model.to(device)
    model.eval()

    return model, checkpoint


baseline_model, baseline_checkpoint = load_baseline_model()

print("\nBaseline MobileNetV2 loaded successfully")

Checkpoint keys:
- epoch
- model_state_dict
- optimizer_state_dict
- scheduler_state_dict
- scaler_state_dict
- best_val_accuracy
- training_history
- num_classes
- image_size
- batch_size
- learning_rate
- model_name

Loaded checkpoint['model_state_dict']
Checkpoint epoch: 9
Best validation accuracy: 0.9969310090842131
Number of classes: 38
Image size: 224
Batch size: 32
Model name: MobileNetV2

Baseline MobileNetV2 loaded successfully


In [25]:
# Cell 09: Define evaluation and complexity counting functions.
# This cell creates helper functions for validation metrics and parameter/MAC counting.

import torch
import torch.nn as nn
import torch_pruning as tp
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm.auto import tqdm

def evaluate_model(model, data_loader, device, desc="Evaluating"):
    model.eval()

    all_preds = []
    all_labels = []
    total_loss = 0.0

    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for images, labels in tqdm(data_loader, desc=desc):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            preds = torch.argmax(outputs, dim=1)

            total_loss += loss.item() * images.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(data_loader.dataset)
    acc = accuracy_score(all_labels, all_preds)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    return {
        "loss": avg_loss,
        "accuracy": acc,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    }


def count_model_complexity(model):
    model.eval()

    example_inputs = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)

    macs, params = tp.utils.count_ops_and_params(
        model,
        example_inputs
    )

    return {
        "macs": float(macs),
        "params": int(params),
        "macs_million": float(macs / 1e6),
        "params_million": float(params / 1e6)
    }

In [26]:
# Cell 10: Verify baseline validation performance before pruning.
# This cell checks baseline validation accuracy and model complexity before applying pruning.

baseline_complexity = count_model_complexity(baseline_model)

baseline_val_metrics = evaluate_model(
    baseline_model,
    val_loader,
    device,
    desc="Baseline validation before pruning"
)

print("\nBaseline complexity:")
for k, v in baseline_complexity.items():
    print(k, ":", v)

print("\nBaseline validation metrics:")
for k, v in baseline_val_metrics.items():
    print(k, ":", v)

Baseline validation before pruning:   0%|          | 0/255 [00:00<?, ?it/s]


Baseline complexity:
macs : 319004966.0
params : 2272550
macs_million : 319.004966
params_million : 2.27255

Baseline validation metrics:
loss : 0.011090856459326292
accuracy : 0.9969310090842131
macro_precision : 0.996324167721049
macro_recall : 0.9967992163745207
macro_f1 : 0.9965513617249061


In [28]:
# Cell 11: Define structured pruning helper functions.
# This cell creates functions to load a fresh baseline model, apply structured pruning, calculate reductions, and save pruned models.

import copy
import json
import torch
import torch.nn as nn
import torch_pruning as tp
from pathlib import Path

def load_fresh_baseline_model():
    """
    Load a fresh copy of the baseline MobileNetV2 model from the best checkpoint.
    This is needed because each pruning candidate must start from the same baseline.
    """
    model = create_mobilenetv2(NUM_CLASSES)

    checkpoint = torch.load(
        BASELINE_MODEL_PATH,
        map_location=device,
        weights_only=False
    )

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint)

    model = model.to(device)
    model.eval()

    return model


def get_ignored_layers(model):
    """
    Tell Torch-Pruning not to prune the final classifier output layer.
    The final output must stay 38 classes.
    """
    ignored_layers = []

    for module in model.modules():
        if isinstance(module, nn.Linear) and module.out_features == NUM_CLASSES:
            ignored_layers.append(module)

    return ignored_layers


def apply_structured_pruning(model, pruning_ratio):
    """
    Apply DepGraph/Torch-Pruning structured pruning.
    This physically removes channels/structures where possible, which is more useful for real parameter/MAC reduction.
    """
    model.eval()

    example_inputs = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
    ignored_layers = get_ignored_layers(model)

    importance = tp.importance.MagnitudeImportance(p=2)

    # Some Torch-Pruning versions support global_pruning and round_to, some may not.
    # round_to=8 is useful for hardware-friendly channel numbers.
    try:
        pruner = tp.pruner.MagnitudePruner(
            model=model,
            example_inputs=example_inputs,
            importance=importance,
            pruning_ratio=pruning_ratio,
            ignored_layers=ignored_layers,
            global_pruning=True,
            round_to=8
        )
    except TypeError:
        pruner = tp.pruner.MagnitudePruner(
            model=model,
            example_inputs=example_inputs,
            importance=importance,
            pruning_ratio=pruning_ratio,
            ignored_layers=ignored_layers
        )

    pruner.step()

    return model


def calculate_reduction(before_value, after_value):
    """
    Calculate reduction percentage.
    """
    return ((before_value - after_value) / before_value) * 100


def save_pruned_full_model(model, save_path, pruning_ratio, best_metrics, complexity, history):
    """
    Save the full structurally pruned model.
    Important: because structured pruning changes the architecture, state_dict alone is not enough for easy loading later.
    """
    model_cpu = copy.deepcopy(model).cpu()

    checkpoint = {
        "model": model_cpu,
        "model_state_dict": model_cpu.state_dict(),
        "pruning_ratio": pruning_ratio,
        "num_classes": NUM_CLASSES,
        "image_size": IMG_SIZE,
        "best_val_metrics": best_metrics,
        "complexity": complexity,
        "history": history,
        "model_name": "Structured Pruned MobileNetV2",
        "important_note": "This is a structurally pruned model. Load with torch.load(..., weights_only=False)."
    }

    torch.save(checkpoint, save_path)

    print("Saved pruned full model:", save_path)

In [30]:
# Cell 12: Define fine-tuning functions.
# This cell creates training and validation logic for each pruned candidate model.

import time
import numpy as np
import torch.optim as optim

def train_one_epoch(model, train_loader, optimizer, criterion, scaler, device, epoch, total_epochs):
    """
    Train the pruned model for one epoch.
    """
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    progress_bar = tqdm(train_loader, desc=f"Training epoch {epoch}/{total_epochs}")

    for images, labels in progress_bar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        preds = torch.argmax(outputs, dim=1)

        running_loss += loss.item() * images.size(0)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

        progress_bar.set_postfix({"loss": loss.item()})

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_acc


def fine_tune_pruned_model(model, train_loader, val_loader, pruning_ratio):
    """
    Fine-tune one pruned candidate model and keep the best validation checkpoint in memory.
    """
    criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS_PER_CANDIDATE
    )

    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    best_val_acc = -1.0
    best_state_dict = None
    best_metrics = None
    history = []

    start_time = time.time()

    for epoch in range(1, EPOCHS_PER_CANDIDATE + 1):
        train_loss, train_acc = train_one_epoch(
            model=model,
            train_loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            scaler=scaler,
            device=device,
            epoch=epoch,
            total_epochs=EPOCHS_PER_CANDIDATE
        )

        val_metrics = evaluate_model(
            model,
            val_loader,
            device,
            desc=f"Validation ratio {pruning_ratio}, epoch {epoch}"
        )

        scheduler.step()

        row = {
            "pruning_ratio": pruning_ratio,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_precision": val_metrics["macro_precision"],
            "val_macro_recall": val_metrics["macro_recall"],
            "val_macro_f1": val_metrics["macro_f1"],
            "learning_rate": optimizer.param_groups[0]["lr"]
        }

        history.append(row)

        print("\nEpoch summary:")
        for k, v in row.items():
            print(k, ":", v)

        if val_metrics["accuracy"] > best_val_acc:
            best_val_acc = val_metrics["accuracy"]
            best_metrics = val_metrics
            best_state_dict = copy.deepcopy(model.state_dict())
            print("New best validation accuracy found:", best_val_acc)

    total_time = time.time() - start_time

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model, best_metrics, history, total_time

In [32]:
# Cell 13: Set pruning configuration for only 5% structured pruning
# This cell configures the experiment to run only one pruning ratio: 5%.

PRUNING_RATIOS = [0.05]

EPOCHS_PER_CANDIDATE = 3
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4

print("Selected pruning ratio:", PRUNING_RATIOS)
print("Epochs:", EPOCHS_PER_CANDIDATE)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

Selected pruning ratio: [0.05]
Epochs: 3
Learning rate: 1e-05
Weight decay: 0.0001


In [33]:
# Cell 14: Define safe structured pruning functions for 5% pruning
# This cell avoids global pruning and round_to=8 because the previous aggressive setup caused high accuracy drop.

import copy
import torch
import torch.nn as nn
import torch_pruning as tp
from pathlib import Path

def load_fresh_baseline_model():
    model = create_mobilenetv2(NUM_CLASSES)

    checkpoint = torch.load(
        BASELINE_MODEL_PATH,
        map_location=device,
        weights_only=False
    )

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint)

    model = model.to(device)
    model.eval()
    return model


def get_ignored_layers(model):
    ignored_layers = []

    for module in model.modules():
        if isinstance(module, nn.Linear) and module.out_features == NUM_CLASSES:
            ignored_layers.append(module)

    return ignored_layers


def apply_safe_structured_pruning(model, pruning_ratio):
    model.eval()

    example_inputs = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
    ignored_layers = get_ignored_layers(model)

    importance = tp.importance.MagnitudeImportance(p=2)

    pruner = tp.pruner.MagnitudePruner(
        model=model,
        example_inputs=example_inputs,
        importance=importance,
        pruning_ratio=pruning_ratio,
        ignored_layers=ignored_layers
    )

    pruner.step()
    return model


def calculate_reduction(before_value, after_value):
    return ((before_value - after_value) / before_value) * 100


def save_pruned_full_model(model, save_path, pruning_ratio, best_metrics, complexity, history):
    model_cpu = copy.deepcopy(model).cpu()

    checkpoint = {
        "model": model_cpu,
        "model_state_dict": model_cpu.state_dict(),
        "pruning_ratio": pruning_ratio,
        "num_classes": NUM_CLASSES,
        "image_size": IMG_SIZE,
        "best_val_metrics": best_metrics,
        "complexity": complexity,
        "history": history,
        "model_name": "5 Percent Safe Structured Pruned MobileNetV2",
        "important_note": "This is a structurally pruned model. Load with torch.load(..., weights_only=False)."
    }

    torch.save(checkpoint, save_path)
    print("Saved pruned full model:", save_path)

In [34]:
# Cell 15: Run only 5% structured pruning and fine-tuning
# This cell prunes the baseline MobileNetV2 by 5%, fine-tunes it for 3 epochs, and saves the result.

import pandas as pd
import torch

pruning_ratio = 0.05

BASELINE_VAL_ACC = baseline_val_metrics["accuracy"]
BASELINE_VAL_F1 = baseline_val_metrics["macro_f1"]
BASELINE_PARAMS = baseline_complexity["params"]
BASELINE_MACS = baseline_complexity["macs"]

print("Baseline validation accuracy:", BASELINE_VAL_ACC)
print("Baseline validation macro F1:", BASELINE_VAL_F1)
print("Baseline parameters:", BASELINE_PARAMS)
print("Baseline MACs:", BASELINE_MACS)

print("\n" + "=" * 100)
print("Starting only 5% safe structured pruning")
print("=" * 100)

torch.cuda.empty_cache()

candidate_model = load_fresh_baseline_model()

before_complexity = count_model_complexity(candidate_model)

candidate_model = apply_safe_structured_pruning(
    model=candidate_model,
    pruning_ratio=pruning_ratio
)

after_pruning_complexity = count_model_complexity(candidate_model)

print("\nComplexity after 5% pruning before fine-tuning:")
for k, v in after_pruning_complexity.items():
    print(k, ":", v)

pre_finetune_metrics = evaluate_model(
    candidate_model,
    val_loader,
    device,
    desc="Validation before fine-tuning 5% pruning"
)

print("\nValidation before fine-tuning:")
for k, v in pre_finetune_metrics.items():
    print(k, ":", v)

candidate_model, best_metrics, history, fine_tune_time = fine_tune_pruned_model(
    model=candidate_model,
    train_loader=train_loader,
    val_loader=val_loader,
    pruning_ratio=pruning_ratio
)

final_complexity = count_model_complexity(candidate_model)

param_reduction = calculate_reduction(BASELINE_PARAMS, final_complexity["params"])
mac_reduction = calculate_reduction(BASELINE_MACS, final_complexity["macs"])

accuracy_drop_percentage_point = (BASELINE_VAL_ACC - best_metrics["accuracy"]) * 100
f1_drop_percentage_point = (BASELINE_VAL_F1 - best_metrics["macro_f1"]) * 100

candidate_model_path = MODEL_DIR / "mobilenetv2_safe_structured_pruned_5_percent_full_model.pth"

save_pruned_full_model(
    model=candidate_model,
    save_path=candidate_model_path,
    pruning_ratio=pruning_ratio,
    best_metrics=best_metrics,
    complexity=final_complexity,
    history=history
)

summary_row = {
    "model_name": "Safe Structured Pruned MobileNetV2 5%",
    "pruning_ratio": pruning_ratio,
    "saved_model_path": str(candidate_model_path),

    "baseline_val_accuracy": BASELINE_VAL_ACC,
    "baseline_val_macro_f1": BASELINE_VAL_F1,
    "baseline_params": BASELINE_PARAMS,
    "baseline_macs": BASELINE_MACS,

    "pre_finetune_val_accuracy": pre_finetune_metrics["accuracy"],
    "pre_finetune_val_macro_f1": pre_finetune_metrics["macro_f1"],

    "best_val_accuracy": best_metrics["accuracy"],
    "best_val_macro_precision": best_metrics["macro_precision"],
    "best_val_macro_recall": best_metrics["macro_recall"],
    "best_val_macro_f1": best_metrics["macro_f1"],
    "best_val_loss": best_metrics["loss"],

    "final_params": final_complexity["params"],
    "final_macs": final_complexity["macs"],
    "final_params_million": final_complexity["params_million"],
    "final_macs_million": final_complexity["macs_million"],

    "parameter_reduction_percent": param_reduction,
    "mac_reduction_percent": mac_reduction,
    "accuracy_drop_percentage_point": accuracy_drop_percentage_point,
    "f1_drop_percentage_point": f1_drop_percentage_point,
    "fine_tune_time_seconds": fine_tune_time,
    "fine_tune_epochs": EPOCHS_PER_CANDIDATE
}

summary_df = pd.DataFrame([summary_row])
history_df = pd.DataFrame(history)

summary_path = RESULTS_DIR / "safe_structured_pruning_5_percent_summary.csv"
history_path = RESULTS_DIR / "safe_structured_pruning_5_percent_history.csv"

summary_df.to_csv(summary_path, index=False)
history_df.to_csv(history_path, index=False)

print("\n5% pruning summary saved:", summary_path)
print("5% pruning history saved:", history_path)

display(summary_df)

Baseline validation accuracy: 0.9969310090842131
Baseline validation macro F1: 0.9965513617249061
Baseline parameters: 2272550
Baseline MACs: 319004966.0

Starting only 5% safe structured pruning

Complexity after 5% pruning before fine-tuning:
macs : 287103338.0
params : 2053526
macs_million : 287.103338
params_million : 2.053526


Validation before fine-tuning 5% pruning:   0%|          | 0/255 [00:00<?, ?it/s]


Validation before fine-tuning:
loss : 1.8604513056413448
accuracy : 0.5114166462067272
macro_precision : 0.697536869601671
macro_recall : 0.5162369524132139
macro_f1 : 0.451008185208387


/tmp/ipykernel_1158/3452586580.py:65: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


Training epoch 1/3:   0%|          | 0/1273 [00:00<?, ?it/s]

...

Training epoch 3/3:   0%|          | 0/1273 [00:00<?, ?it/s]

/tmp/ipykernel_1158/3452586580.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Validation ratio 0.05, epoch 3:   0%|          | 0/255 [00:00<?, ?it/s]


Epoch summary:
pruning_ratio : 0.05
epoch : 3
train_loss : 0.0202233183099441
train_accuracy : 0.9944509919465724
val_loss : 0.021570975869201366
val_accuracy : 0.9936164988951632
val_macro_precision : 0.9919551858363098
val_macro_recall : 0.9903100919397236
val_macro_f1 : 0.9911028713065136
learning_rate : 0.0
Saved pruned full model: /content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/models/mobilenetv2_safe_structured_pruned_5_percent_full_model.pth

5% pruning summary saved: /content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/results/safe_structured_pruning_5_percent_summary.csv
5% pruning history saved: /content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/results/safe_structured_pruning_5_percent_history.csv


,model_name,pruning_ratio,saved_model_path,baseline_val_accuracy,baseline_val_macro_f1,baseline_params,baseline_macs,pre_finetune_val_accuracy,pre_finetune_val_macro_f1,best_val_accuracy,...,final_params,final_macs,final_params_million,final_macs_million,parameter_reduction_percent,mac_reduction_percent,accuracy_drop_percentage_point,f1_drop_percentage_point,fine_tune_time_seconds,fine_tune_epochs
0,Safe Structured Pruned MobileNetV2 5%,0.05,/content/drive/MyDrive/Thesis/thesis_project/0...,0.996931,0.996551,2272550,319004966.0,0.511417,0.451008,0.993616,...,2053526,287103338.0,2.053526,287.103338,9.637808,10.000355,0.331451,0.544307,636.841859,3


In [35]:
# Cell 17: Set exploratory pruning configuration for 6% and 7%
# This cell prepares two additional pruning ratios to test whether more compression is possible than 5%.

EXPLORATORY_PRUNING_RATIOS = [0.06, 0.07]

EPOCHS_PER_CANDIDATE = 3
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4

# Acceptance rules for saving model
MIN_ACCEPTABLE_VAL_ACCURACY = 0.9900
MAX_ACCEPTABLE_ACC_DROP_PERCENTAGE_POINT = 1.0
MAX_ACCEPTABLE_F1_DROP_PERCENTAGE_POINT = 1.0

print("Exploratory pruning ratios:", EXPLORATORY_PRUNING_RATIOS)
print("Epochs per candidate:", EPOCHS_PER_CANDIDATE)
print("Minimum acceptable validation accuracy:", MIN_ACCEPTABLE_VAL_ACCURACY)
print("Maximum acceptable accuracy drop:", MAX_ACCEPTABLE_ACC_DROP_PERCENTAGE_POINT, "percentage point")
print("Maximum acceptable F1 drop:", MAX_ACCEPTABLE_F1_DROP_PERCENTAGE_POINT, "percentage point")

Exploratory pruning ratios: [0.06, 0.07]
Epochs per candidate: 3
Minimum acceptable validation accuracy: 0.99
Maximum acceptable accuracy drop: 1.0 percentage point
Maximum acceptable F1 drop: 1.0 percentage point


In [36]:
# Cell 18: Run 6% and 7% exploratory pruning and save only acceptable models
# This cell fine-tunes 6% and 7% pruned models, evaluates them, and saves only if they meet acceptance rules.

import pandas as pd
import torch
from pathlib import Path

exploratory_summary = []
exploratory_history = []

BASELINE_VAL_ACC = baseline_val_metrics["accuracy"]
BASELINE_VAL_F1 = baseline_val_metrics["macro_f1"]
BASELINE_PARAMS = baseline_complexity["params"]
BASELINE_MACS = baseline_complexity["macs"]

print("Baseline validation accuracy:", BASELINE_VAL_ACC)
print("Baseline validation macro F1:", BASELINE_VAL_F1)
print("Baseline parameters:", BASELINE_PARAMS)
print("Baseline MACs:", BASELINE_MACS)

for pruning_ratio in EXPLORATORY_PRUNING_RATIOS:
    print("\n" + "=" * 100)
    print(f"Starting exploratory safe structured pruning: {int(pruning_ratio * 100)}%")
    print("=" * 100)

    torch.cuda.empty_cache()

    # Load fresh baseline model for fair comparison
    candidate_model = load_fresh_baseline_model()

    # Apply safe structured pruning
    candidate_model = apply_safe_structured_pruning(
        model=candidate_model,
        pruning_ratio=pruning_ratio
    )

    after_pruning_complexity = count_model_complexity(candidate_model)

    print("\nComplexity after pruning before fine-tuning:")
    for k, v in after_pruning_complexity.items():
        print(k, ":", v)

    # Check validation performance immediately after pruning
    pre_finetune_metrics = evaluate_model(
        candidate_model,
        val_loader,
        device,
        desc=f"Validation before fine-tuning {int(pruning_ratio * 100)}% pruning"
    )

    print("\nValidation before fine-tuning:")
    for k, v in pre_finetune_metrics.items():
        print(k, ":", v)

    # Fine-tune pruned model
    candidate_model, best_metrics, history, fine_tune_time = fine_tune_pruned_model(
        model=candidate_model,
        train_loader=train_loader,
        val_loader=val_loader,
        pruning_ratio=pruning_ratio
    )

    final_complexity = count_model_complexity(candidate_model)

    param_reduction = calculate_reduction(BASELINE_PARAMS, final_complexity["params"])
    mac_reduction = calculate_reduction(BASELINE_MACS, final_complexity["macs"])

    accuracy_drop_percentage_point = (BASELINE_VAL_ACC - best_metrics["accuracy"]) * 100
    f1_drop_percentage_point = (BASELINE_VAL_F1 - best_metrics["macro_f1"]) * 100

    is_acceptable = (
        best_metrics["accuracy"] >= MIN_ACCEPTABLE_VAL_ACCURACY
        and accuracy_drop_percentage_point <= MAX_ACCEPTABLE_ACC_DROP_PERCENTAGE_POINT
        and f1_drop_percentage_point <= MAX_ACCEPTABLE_F1_DROP_PERCENTAGE_POINT
    )

    ratio_tag = int(pruning_ratio * 100)

    saved_model_path = ""

    if is_acceptable:
        candidate_model_path = MODEL_DIR / f"mobilenetv2_safe_structured_pruned_{ratio_tag}_percent_full_model.pth"

        save_pruned_full_model(
            model=candidate_model,
            save_path=candidate_model_path,
            pruning_ratio=pruning_ratio,
            best_metrics=best_metrics,
            complexity=final_complexity,
            history=history
        )

        saved_model_path = str(candidate_model_path)

        print(f"\n✅ {ratio_tag}% pruning accepted and model saved.")
        print("Saved model path:", saved_model_path)

    else:
        print(f"\n❌ {ratio_tag}% pruning rejected. Model not saved.")
        print("Reason: validation accuracy/F1 drop exceeded acceptable threshold.")

    summary_row = {
        "model_name": f"Exploratory Safe Structured Pruned MobileNetV2 {ratio_tag}%",
        "pruning_ratio": pruning_ratio,
        "accepted_for_saving": is_acceptable,
        "saved_model_path": saved_model_path,

        "baseline_val_accuracy": BASELINE_VAL_ACC,
        "baseline_val_macro_f1": BASELINE_VAL_F1,
        "baseline_params": BASELINE_PARAMS,
        "baseline_macs": BASELINE_MACS,

        "pre_finetune_val_accuracy": pre_finetune_metrics["accuracy"],
        "pre_finetune_val_macro_f1": pre_finetune_metrics["macro_f1"],

        "best_val_accuracy": best_metrics["accuracy"],
        "best_val_macro_precision": best_metrics["macro_precision"],
        "best_val_macro_recall": best_metrics["macro_recall"],
        "best_val_macro_f1": best_metrics["macro_f1"],
        "best_val_loss": best_metrics["loss"],

        "final_params": final_complexity["params"],
        "final_macs": final_complexity["macs"],
        "final_params_million": final_complexity["params_million"],
        "final_macs_million": final_complexity["macs_million"],

        "parameter_reduction_percent": param_reduction,
        "mac_reduction_percent": mac_reduction,
        "accuracy_drop_percentage_point": accuracy_drop_percentage_point,
        "f1_drop_percentage_point": f1_drop_percentage_point,
        "fine_tune_time_seconds": fine_tune_time,
        "fine_tune_epochs": EPOCHS_PER_CANDIDATE
    }

    exploratory_summary.append(summary_row)

    for h in history:
        h["exploratory_pruning_ratio"] = pruning_ratio
        exploratory_history.append(h)

    # Save result summaries after each candidate
    exploratory_summary_df = pd.DataFrame(exploratory_summary)
    exploratory_history_df = pd.DataFrame(exploratory_history)

    exploratory_summary_path = RESULTS_DIR / "exploratory_pruning_6_7_percent_summary.csv"
    exploratory_history_path = RESULTS_DIR / "exploratory_pruning_6_7_percent_history.csv"

    exploratory_summary_df.to_csv(exploratory_summary_path, index=False)
    exploratory_history_df.to_csv(exploratory_history_path, index=False)

    print("\nExploratory summary saved:", exploratory_summary_path)
    print("Exploratory history saved:", exploratory_history_path)

print("\nExploratory 6% and 7% pruning completed.")
display(pd.DataFrame(exploratory_summary))

Baseline validation accuracy: 0.9969310090842131
Baseline validation macro F1: 0.9965513617249061
Baseline parameters: 2272550
Baseline MACs: 319004966.0

Starting exploratory safe structured pruning: 6%

Complexity after pruning before fine-tuning:
macs : 282839697.0
params : 2009613
macs_million : 282.839697
params_million : 2.009613


Validation before fine-tuning 6% pruning:   0%|          | 0/255 [00:00<?, ?it/s]


Validation before fine-tuning:
loss : 2.638965572972512
accuracy : 0.3246992388902529
macro_precision : 0.49981466359917076
macro_recall : 0.3572616546383219
macro_f1 : 0.24117932898695119


/tmp/ipykernel_1158/3452586580.py:65: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


Training epoch 1/3:   0%|          | 0/1273 [00:00<?, ?it/s]

...

Training epoch 3/3:   0%|          | 0/1273 [00:00<?, ?it/s]

/tmp/ipykernel_1158/3452586580.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Validation ratio 0.07, epoch 3:   0%|          | 0/255 [00:00<?, ?it/s]


Epoch summary:
pruning_ratio : 0.07
epoch : 3
train_loss : 0.03148024378297219
train_accuracy : 0.991209978393243
val_loss : 0.026416268748917893
val_accuracy : 0.9906702676160078
val_macro_precision : 0.9891730635979836
val_macro_recall : 0.9847134539770968
val_macro_f1 : 0.9868256224650543
learning_rate : 0.0
New best validation accuracy found: 0.9906702676160078
Saved pruned full model: /content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/models/mobilenetv2_safe_structured_pruned_7_percent_full_model.pth

✅ 7% pruning accepted and model saved.
Saved model path: /content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/models/mobilenetv2_safe_structured_pruned_7_percent_full_model.pth

Exploratory summary saved: /content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/results/exploratory_pruning_6_7_percent_summary.csv
Exploratory history saved: /content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/results

,model_name,pruning_ratio,accepted_for_saving,saved_model_path,baseline_val_accuracy,baseline_val_macro_f1,baseline_params,baseline_macs,pre_finetune_val_accuracy,pre_finetune_val_macro_f1,...,final_params,final_macs,final_params_million,final_macs_million,parameter_reduction_percent,mac_reduction_percent,accuracy_drop_percentage_point,f1_drop_percentage_point,fine_tune_time_seconds,fine_tune_epochs
0,Exploratory Safe Structured Pruned MobileNetV2 6%,0.06,True,/content/drive/MyDrive/Thesis/thesis_project/0...,0.996931,0.996551,2272550,319004966.0,0.324699,0.241179,...,2009613,282839697.0,2.009613,282.839697,11.570130,11.336898,0.380555,0.569397,626.013108,3
1,Exploratory Safe Structured Pruned MobileNetV2 7%,0.07,True,/content/drive/MyDrive/Thesis/thesis_project/0...,0.996931,0.996551,2272550,319004966.0,0.157255,0.068243,...,1964810,274973380.0,1.964810,274.973380,13.541616,13.802790,0.626074,0.972574,656.563878,3


In [37]:
# Cell 21: Set exploratory pruning configuration for 8% and 9%
# This cell prepares two additional pruning ratios and will save both models regardless of accuracy.

EXPLORATORY_PRUNING_RATIOS_8_9 = [0.08, 0.09]

EPOCHS_PER_CANDIDATE = 3
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4

print("Exploratory pruning ratios:", EXPLORATORY_PRUNING_RATIOS_8_9)
print("Epochs per candidate:", EPOCHS_PER_CANDIDATE)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Save policy: Save all 8% and 9% models regardless of validation accuracy.")

Exploratory pruning ratios: [0.08, 0.09]
Epochs per candidate: 3
Learning rate: 1e-05
Weight decay: 0.0001
Save policy: Save all 8% and 9% models regardless of validation accuracy.


In [38]:
# Cell 22: Run 8% and 9% exploratory pruning and save all models
# This cell fine-tunes 8% and 9% pruned models, evaluates them, and saves every model for later comparison.

import pandas as pd
import torch
from pathlib import Path

exploratory_summary_8_9 = []
exploratory_history_8_9 = []

BASELINE_VAL_ACC = baseline_val_metrics["accuracy"]
BASELINE_VAL_F1 = baseline_val_metrics["macro_f1"]
BASELINE_PARAMS = baseline_complexity["params"]
BASELINE_MACS = baseline_complexity["macs"]

print("Baseline validation accuracy:", BASELINE_VAL_ACC)
print("Baseline validation macro F1:", BASELINE_VAL_F1)
print("Baseline parameters:", BASELINE_PARAMS)
print("Baseline MACs:", BASELINE_MACS)

for pruning_ratio in EXPLORATORY_PRUNING_RATIOS_8_9:
    ratio_tag = int(pruning_ratio * 100)

    print("\n" + "=" * 100)
    print(f"Starting exploratory safe structured pruning: {ratio_tag}%")
    print("=" * 100)

    torch.cuda.empty_cache()

    # Load fresh baseline model for fair comparison
    candidate_model = load_fresh_baseline_model()

    # Apply safe structured pruning
    candidate_model = apply_safe_structured_pruning(
        model=candidate_model,
        pruning_ratio=pruning_ratio
    )

    after_pruning_complexity = count_model_complexity(candidate_model)

    print("\nComplexity after pruning before fine-tuning:")
    for k, v in after_pruning_complexity.items():
        print(k, ":", v)

    # Validation immediately after pruning
    pre_finetune_metrics = evaluate_model(
        candidate_model,
        val_loader,
        device,
        desc=f"Validation before fine-tuning {ratio_tag}% pruning"
    )

    print("\nValidation before fine-tuning:")
    for k, v in pre_finetune_metrics.items():
        print(k, ":", v)

    # Fine-tune pruned model
    candidate_model, best_metrics, history, fine_tune_time = fine_tune_pruned_model(
        model=candidate_model,
        train_loader=train_loader,
        val_loader=val_loader,
        pruning_ratio=pruning_ratio
    )

    final_complexity = count_model_complexity(candidate_model)

    param_reduction = calculate_reduction(BASELINE_PARAMS, final_complexity["params"])
    mac_reduction = calculate_reduction(BASELINE_MACS, final_complexity["macs"])

    accuracy_drop_percentage_point = (BASELINE_VAL_ACC - best_metrics["accuracy"]) * 100
    f1_drop_percentage_point = (BASELINE_VAL_F1 - best_metrics["macro_f1"]) * 100

    # Save every model regardless of result
    candidate_model_path = MODEL_DIR / f"mobilenetv2_safe_structured_pruned_{ratio_tag}_percent_full_model.pth"

    save_pruned_full_model(
        model=candidate_model,
        save_path=candidate_model_path,
        pruning_ratio=pruning_ratio,
        best_metrics=best_metrics,
        complexity=final_complexity,
        history=history
    )

    saved_model_path = str(candidate_model_path)

    print(f"\n✅ {ratio_tag}% pruning model saved.")
    print("Saved model path:", saved_model_path)

    # Add a flag for later final model decision
    if accuracy_drop_percentage_point <= 1.0 and f1_drop_percentage_point <= 1.0:
        candidate_quality_note = "Strong candidate: accuracy and F1 drop within 1 percentage point."
    elif accuracy_drop_percentage_point <= 2.0 and f1_drop_percentage_point <= 2.0:
        candidate_quality_note = "Possible trade-off candidate: moderate accuracy/F1 drop."
    else:
        candidate_quality_note = "Weak candidate: high accuracy/F1 drop; keep for comparison only."

    summary_row = {
        "model_name": f"Exploratory Safe Structured Pruned MobileNetV2 {ratio_tag}%",
        "pruning_ratio": pruning_ratio,
        "saved_for_comparison": True,
        "candidate_quality_note": candidate_quality_note,
        "saved_model_path": saved_model_path,

        "baseline_val_accuracy": BASELINE_VAL_ACC,
        "baseline_val_macro_f1": BASELINE_VAL_F1,
        "baseline_params": BASELINE_PARAMS,
        "baseline_macs": BASELINE_MACS,

        "pre_finetune_val_accuracy": pre_finetune_metrics["accuracy"],
        "pre_finetune_val_macro_f1": pre_finetune_metrics["macro_f1"],

        "best_val_accuracy": best_metrics["accuracy"],
        "best_val_macro_precision": best_metrics["macro_precision"],
        "best_val_macro_recall": best_metrics["macro_recall"],
        "best_val_macro_f1": best_metrics["macro_f1"],
        "best_val_loss": best_metrics["loss"],

        "final_params": final_complexity["params"],
        "final_macs": final_complexity["macs"],
        "final_params_million": final_complexity["params_million"],
        "final_macs_million": final_complexity["macs_million"],

        "parameter_reduction_percent": param_reduction,
        "mac_reduction_percent": mac_reduction,
        "accuracy_drop_percentage_point": accuracy_drop_percentage_point,
        "f1_drop_percentage_point": f1_drop_percentage_point,
        "fine_tune_time_seconds": fine_tune_time,
        "fine_tune_epochs": EPOCHS_PER_CANDIDATE
    }

    exploratory_summary_8_9.append(summary_row)

    for h in history:
        h["exploratory_pruning_ratio"] = pruning_ratio
        h["model_saved_for_comparison"] = True
        exploratory_history_8_9.append(h)

    # Save result summaries after each candidate
    exploratory_summary_df_8_9 = pd.DataFrame(exploratory_summary_8_9)
    exploratory_history_df_8_9 = pd.DataFrame(exploratory_history_8_9)

    exploratory_summary_path_8_9 = RESULTS_DIR / "exploratory_pruning_8_9_percent_summary.csv"
    exploratory_history_path_8_9 = RESULTS_DIR / "exploratory_pruning_8_9_percent_history.csv"

    exploratory_summary_df_8_9.to_csv(exploratory_summary_path_8_9, index=False)
    exploratory_history_df_8_9.to_csv(exploratory_history_path_8_9, index=False)

    print("\nExploratory 8/9 summary saved:", exploratory_summary_path_8_9)
    print("Exploratory 8/9 history saved:", exploratory_history_path_8_9)

print("\nExploratory 8% and 9% pruning completed.")
display(pd.DataFrame(exploratory_summary_8_9))

Baseline validation accuracy: 0.9969310090842131
Baseline validation macro F1: 0.9965513617249061
Baseline parameters: 2272550
Baseline MACs: 319004966.0

Starting exploratory safe structured pruning: 8%

Complexity after pruning before fine-tuning:
macs : 270677644.0
params : 1927286
macs_million : 270.677644
params_million : 1.927286


Validation before fine-tuning 8% pruning:   0%|          | 0/255 [00:00<?, ?it/s]


Validation before fine-tuning:
loss : 3.4043151270456407
accuracy : 0.1492757181438743
macro_precision : 0.15248029166436633
macro_recall : 0.12751598919064014
macro_f1 : 0.06585114360254801


/tmp/ipykernel_1158/3452586580.py:65: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


Training epoch 1/3:   0%|          | 0/1273 [00:00<?, ?it/s]

...

Training epoch 3/3:   0%|          | 0/1273 [00:00<?, ?it/s]

/tmp/ipykernel_1158/3452586580.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Validation ratio 0.09, epoch 3:   0%|          | 0/255 [00:00<?, ?it/s]


Epoch summary:
pruning_ratio : 0.09
epoch : 3
train_loss : 0.04420191162263825
train_accuracy : 0.9872814771164801
val_loss : 0.03795980138755858
val_accuracy : 0.9885833537932728
val_macro_precision : 0.9849147114640521
val_macro_recall : 0.9820741552577159
val_macro_f1 : 0.9834242384722838
learning_rate : 0.0
New best validation accuracy found: 0.9885833537932728
Saved pruned full model: /content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/models/mobilenetv2_safe_structured_pruned_9_percent_full_model.pth

✅ 9% pruning model saved.
Saved model path: /content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/models/mobilenetv2_safe_structured_pruned_9_percent_full_model.pth

Exploratory 8/9 summary saved: /content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/results/exploratory_pruning_8_9_percent_summary.csv
Exploratory 8/9 history saved: /content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/results/expl

,model_name,pruning_ratio,saved_for_comparison,candidate_quality_note,saved_model_path,baseline_val_accuracy,baseline_val_macro_f1,baseline_params,baseline_macs,pre_finetune_val_accuracy,...,final_params,final_macs,final_params_million,final_macs_million,parameter_reduction_percent,mac_reduction_percent,accuracy_drop_percentage_point,f1_drop_percentage_point,fine_tune_time_seconds,fine_tune_epochs
0,Exploratory Safe Structured Pruned MobileNetV2 8%,0.08,True,Strong candidate: accuracy and F1 drop within ...,/content/drive/MyDrive/Thesis/thesis_project/0...,0.996931,0.996551,2272550,319004966.0,0.149276,...,1927286,270677644.0,1.927286,270.677644,15.192801,15.149395,0.527866,0.828391,641.746140,3
1,Exploratory Safe Structured Pruned MobileNetV2 9%,0.09,True,Possible trade-off candidate: moderate accurac...,/content/drive/MyDrive/Thesis/thesis_project/0...,0.996931,0.996551,2272550,319004966.0,0.061994,...,1886930,265222519.0,1.886930,265.222519,16.968604,16.859439,0.834766,1.312712,632.140243,3


In [39]:
# Cell 25: Set exploratory pruning configuration for 10% and 11%
# This cell prepares two additional safe structured pruning ratios and will save both models for later comparison.

EXPLORATORY_PRUNING_RATIOS_10_11 = [0.10, 0.11]

EPOCHS_PER_CANDIDATE = 3
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4

print("Exploratory pruning ratios:", EXPLORATORY_PRUNING_RATIOS_10_11)
print("Epochs per candidate:", EPOCHS_PER_CANDIDATE)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Save policy: Save all 10% and 11% models regardless of validation accuracy.")

Exploratory pruning ratios: [0.1, 0.11]
Epochs per candidate: 3
Learning rate: 1e-05
Weight decay: 0.0001
Save policy: Save all 10% and 11% models regardless of validation accuracy.


In [40]:
# Cell 26: Run 10% and 11% exploratory pruning and save all models
# This cell fine-tunes 10% and 11% pruned models, evaluates them, and saves every model for later comparison.

import pandas as pd
import torch
from pathlib import Path

exploratory_summary_10_11 = []
exploratory_history_10_11 = []

BASELINE_VAL_ACC = baseline_val_metrics["accuracy"]
BASELINE_VAL_F1 = baseline_val_metrics["macro_f1"]
BASELINE_PARAMS = baseline_complexity["params"]
BASELINE_MACS = baseline_complexity["macs"]

print("Baseline validation accuracy:", BASELINE_VAL_ACC)
print("Baseline validation macro F1:", BASELINE_VAL_F1)
print("Baseline parameters:", BASELINE_PARAMS)
print("Baseline MACs:", BASELINE_MACS)

for pruning_ratio in EXPLORATORY_PRUNING_RATIOS_10_11:
    ratio_tag = int(pruning_ratio * 100)

    print("\n" + "=" * 100)
    print(f"Starting exploratory safe structured pruning: {ratio_tag}%")
    print("=" * 100)

    torch.cuda.empty_cache()

    # Load fresh baseline model for fair comparison
    candidate_model = load_fresh_baseline_model()

    # Apply safe structured pruning
    candidate_model = apply_safe_structured_pruning(
        model=candidate_model,
        pruning_ratio=pruning_ratio
    )

    after_pruning_complexity = count_model_complexity(candidate_model)

    print("\nComplexity after pruning before fine-tuning:")
    for k, v in after_pruning_complexity.items():
        print(k, ":", v)

    # Validation immediately after pruning
    pre_finetune_metrics = evaluate_model(
        candidate_model,
        val_loader,
        device,
        desc=f"Validation before fine-tuning {ratio_tag}% pruning"
    )

    print("\nValidation before fine-tuning:")
    for k, v in pre_finetune_metrics.items():
        print(k, ":", v)

    # Fine-tune pruned model
    candidate_model, best_metrics, history, fine_tune_time = fine_tune_pruned_model(
        model=candidate_model,
        train_loader=train_loader,
        val_loader=val_loader,
        pruning_ratio=pruning_ratio
    )

    final_complexity = count_model_complexity(candidate_model)

    param_reduction = calculate_reduction(BASELINE_PARAMS, final_complexity["params"])
    mac_reduction = calculate_reduction(BASELINE_MACS, final_complexity["macs"])

    accuracy_drop_percentage_point = (BASELINE_VAL_ACC - best_metrics["accuracy"]) * 100
    f1_drop_percentage_point = (BASELINE_VAL_F1 - best_metrics["macro_f1"]) * 100

    # Save every model regardless of result
    candidate_model_path = MODEL_DIR / f"mobilenetv2_safe_structured_pruned_{ratio_tag}_percent_full_model.pth"

    save_pruned_full_model(
        model=candidate_model,
        save_path=candidate_model_path,
        pruning_ratio=pruning_ratio,
        best_metrics=best_metrics,
        complexity=final_complexity,
        history=history
    )

    saved_model_path = str(candidate_model_path)

    print(f"\n✅ {ratio_tag}% pruning model saved.")
    print("Saved model path:", saved_model_path)

    # Quality note for later decision
    if accuracy_drop_percentage_point <= 1.0 and f1_drop_percentage_point <= 1.0:
        candidate_quality_note = "Strong candidate: accuracy and F1 drop within 1 percentage point."
    elif accuracy_drop_percentage_point <= 2.0 and f1_drop_percentage_point <= 2.0:
        candidate_quality_note = "Possible trade-off candidate: moderate accuracy/F1 drop."
    else:
        candidate_quality_note = "Weak candidate: high accuracy/F1 drop; keep for comparison only."

    summary_row = {
        "model_name": f"Exploratory Safe Structured Pruned MobileNetV2 {ratio_tag}%",
        "pruning_ratio": pruning_ratio,
        "saved_for_comparison": True,
        "candidate_quality_note": candidate_quality_note,
        "saved_model_path": saved_model_path,

        "baseline_val_accuracy": BASELINE_VAL_ACC,
        "baseline_val_macro_f1": BASELINE_VAL_F1,
        "baseline_params": BASELINE_PARAMS,
        "baseline_macs": BASELINE_MACS,

        "pre_finetune_val_accuracy": pre_finetune_metrics["accuracy"],
        "pre_finetune_val_macro_f1": pre_finetune_metrics["macro_f1"],

        "best_val_accuracy": best_metrics["accuracy"],
        "best_val_macro_precision": best_metrics["macro_precision"],
        "best_val_macro_recall": best_metrics["macro_recall"],
        "best_val_macro_f1": best_metrics["macro_f1"],
        "best_val_loss": best_metrics["loss"],

        "final_params": final_complexity["params"],
        "final_macs": final_complexity["macs"],
        "final_params_million": final_complexity["params_million"],
        "final_macs_million": final_complexity["macs_million"],

        "parameter_reduction_percent": param_reduction,
        "mac_reduction_percent": mac_reduction,
        "accuracy_drop_percentage_point": accuracy_drop_percentage_point,
        "f1_drop_percentage_point": f1_drop_percentage_point,
        "fine_tune_time_seconds": fine_tune_time,
        "fine_tune_epochs": EPOCHS_PER_CANDIDATE
    }

    exploratory_summary_10_11.append(summary_row)

    for h in history:
        h["exploratory_pruning_ratio"] = pruning_ratio
        h["model_saved_for_comparison"] = True
        exploratory_history_10_11.append(h)

    # Save result summaries after each candidate
    exploratory_summary_df_10_11 = pd.DataFrame(exploratory_summary_10_11)
    exploratory_history_df_10_11 = pd.DataFrame(exploratory_history_10_11)

    exploratory_summary_path_10_11 = RESULTS_DIR / "exploratory_pruning_10_11_percent_summary.csv"
    exploratory_history_path_10_11 = RESULTS_DIR / "exploratory_pruning_10_11_percent_history.csv"

    exploratory_summary_df_10_11.to_csv(exploratory_summary_path_10_11, index=False)
    exploratory_history_df_10_11.to_csv(exploratory_history_path_10_11, index=False)

    print("\nExploratory 10/11 summary saved:", exploratory_summary_path_10_11)
    print("Exploratory 10/11 history saved:", exploratory_history_path_10_11)

print("\nExploratory 10% and 11% pruning completed.")
display(pd.DataFrame(exploratory_summary_10_11))

Baseline validation accuracy: 0.9969310090842131
Baseline validation macro F1: 0.9965513617249061
Baseline parameters: 2272550
Baseline MACs: 319004966.0

Starting exploratory safe structured pruning: 10%

Complexity after pruning before fine-tuning:
macs : 259254010.0
params : 1849318
macs_million : 259.25401
params_million : 1.849318


Validation before fine-tuning 10% pruning:   0%|          | 0/255 [00:00<?, ?it/s]


Validation before fine-tuning:
loss : 3.749306792976404
accuracy : 0.04136999754480727
macro_precision : 0.02154275975389373
macro_recall : 0.038395146541498344
macro_f1 : 0.009643561586200047


/tmp/ipykernel_1158/3452586580.py:65: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


Training epoch 1/3:   0%|          | 0/1273 [00:00<?, ?it/s]

...

Training epoch 3/3:   0%|          | 0/1273 [00:00<?, ?it/s]

/tmp/ipykernel_1158/3452586580.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Validation ratio 0.11, epoch 3:   0%|          | 0/255 [00:00<?, ?it/s]


Epoch summary:
pruning_ratio : 0.11
epoch : 3
train_loss : 0.06008142677207659
train_accuracy : 0.9831319976428993
val_loss : 0.047960480862886366
val_accuracy : 0.9862509206972747
val_macro_precision : 0.9837854443512238
val_macro_recall : 0.9818269836199642
val_macro_f1 : 0.9826975305474518
learning_rate : 0.0
New best validation accuracy found: 0.9862509206972747
Saved pruned full model: /content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/models/mobilenetv2_safe_structured_pruned_11_percent_full_model.pth

✅ 11% pruning model saved.
Saved model path: /content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/models/mobilenetv2_safe_structured_pruned_11_percent_full_model.pth

Exploratory 10/11 summary saved: /content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/results/exploratory_pruning_10_11_percent_summary.csv
Exploratory 10/11 history saved: /content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/re

,model_name,pruning_ratio,saved_for_comparison,candidate_quality_note,saved_model_path,baseline_val_accuracy,baseline_val_macro_f1,baseline_params,baseline_macs,pre_finetune_val_accuracy,...,final_params,final_macs,final_params_million,final_macs_million,parameter_reduction_percent,mac_reduction_percent,accuracy_drop_percentage_point,f1_drop_percentage_point,fine_tune_time_seconds,fine_tune_epochs
0,Exploratory Safe Structured Pruned MobileNetV2...,0.10,True,Possible trade-off candidate: moderate accurac...,/content/drive/MyDrive/Thesis/thesis_project/0...,0.996931,0.996551,2272550,319004966.0,0.041370,...,1849318,259254010.0,1.849318,259.254010,18.623661,18.730416,1.092561,1.630674,646.742426,3
1,Exploratory Safe Structured Pruned MobileNetV2...,0.11,True,Possible trade-off candidate: moderate accurac...,/content/drive/MyDrive/Thesis/thesis_project/0...,0.996931,0.996551,2272550,319004966.0,0.012644,...,1805093,254701367.0,1.805093,254.701367,20.569712,20.157554,1.068009,1.385383,643.148116,3


In [43]:
# Cell 27A: Inspect saved pruning model files and result CSV files
# This cell lists all pruning-related .pth and .csv files to identify missing or saved outputs.

from pathlib import Path

print("MODEL_DIR:", MODEL_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

print("\nSaved pruning model files:")
model_files = sorted(MODEL_DIR.glob("*pruned*percent*full_model.pth"))
for f in model_files:
    print("-", f.name)

print("\nSaved result CSV files:")
csv_files = sorted(RESULTS_DIR.glob("*.csv"))
for f in csv_files:
    print("-", f.name)

MODEL_DIR: /content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/models
RESULTS_DIR: /content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/results

Saved pruning model files:
- mobilenetv2_safe_structured_pruned_10_percent_full_model.pth
- mobilenetv2_safe_structured_pruned_11_percent_full_model.pth
- mobilenetv2_safe_structured_pruned_5_percent_full_model.pth
- mobilenetv2_safe_structured_pruned_6_percent_full_model.pth
- mobilenetv2_safe_structured_pruned_7_percent_full_model.pth
- mobilenetv2_safe_structured_pruned_8_percent_full_model.pth
- mobilenetv2_safe_structured_pruned_9_percent_full_model.pth

Saved result CSV files:
- exploratory_pruning_10_percent_history.csv
- exploratory_pruning_10_percent_summary.csv
- exploratory_pruning_11_percent_history.csv
- exploratory_pruning_11_percent_summary.csv
- exploratory_pruning_6_percent_history.csv
- exploratory_pruning_6_percent_summary.csv
- exploratory_pruning_7_percent_history.csv
- explorat

In [44]:
# Cell 27B: Create final pruning comparison table from individually saved summary CSV files
# This cell loads 5%, 6%, 7%, 8%, 9%, 10%, and 11% pruning summaries and combines them into one comparison table.

import pandas as pd
from pathlib import Path

summary_files = [
    RESULTS_DIR / "safe_structured_pruning_5_percent_summary.csv",
    RESULTS_DIR / "exploratory_pruning_6_percent_summary.csv",
    RESULTS_DIR / "exploratory_pruning_7_percent_summary.csv",
    RESULTS_DIR / "exploratory_pruning_8_percent_summary.csv",
    RESULTS_DIR / "exploratory_pruning_9_percent_summary.csv",
    RESULTS_DIR / "exploratory_pruning_10_percent_summary.csv",
    RESULTS_DIR / "exploratory_pruning_11_percent_summary.csv",
]

summary_dfs = []

for file_path in summary_files:
    if file_path.exists():
        df = pd.read_csv(file_path)
        summary_dfs.append(df)
        print("Loaded:", file_path.name)
    else:
        print("Missing:", file_path.name)

if len(summary_dfs) == 0:
    raise FileNotFoundError("No pruning summary CSV files found.")

comparison_df = pd.concat(summary_dfs, ignore_index=True)

# Normalize saved flag
if "saved_for_comparison" not in comparison_df.columns:
    comparison_df["saved_for_comparison"] = True
else:
    comparison_df["saved_for_comparison"] = comparison_df["saved_for_comparison"].fillna(True)

if "accepted_for_saving" in comparison_df.columns:
    comparison_df["saved_for_comparison"] = comparison_df["saved_for_comparison"] | comparison_df["accepted_for_saving"].fillna(False)

# Add candidate quality note if missing
def quality_note(row):
    acc_drop = row.get("accuracy_drop_percentage_point", None)
    f1_drop = row.get("f1_drop_percentage_point", None)

    if pd.isna(acc_drop) or pd.isna(f1_drop):
        return "Incomplete metrics."
    elif acc_drop <= 1.0 and f1_drop <= 1.0:
        return "Strong candidate: accuracy and F1 drop within 1 percentage point."
    elif acc_drop <= 2.0 and f1_drop <= 2.0:
        return "Possible trade-off candidate: moderate accuracy/F1 drop."
    else:
        return "Weak candidate: high accuracy/F1 drop; keep for comparison only."

comparison_df["candidate_quality_note"] = comparison_df.apply(quality_note, axis=1)

comparison_output_path = RESULTS_DIR / "pruning_5_6_7_8_9_10_11_final_comparison.csv"
comparison_df.to_csv(comparison_output_path, index=False)

display_columns = [
    "model_name",
    "pruning_ratio",
    "saved_for_comparison",
    "candidate_quality_note",
    "best_val_accuracy",
    "best_val_macro_f1",
    "accuracy_drop_percentage_point",
    "f1_drop_percentage_point",
    "parameter_reduction_percent",
    "mac_reduction_percent",
    "final_params_million",
    "final_macs_million",
    "saved_model_path"
]

display_columns = [col for col in display_columns if col in comparison_df.columns]

print("\nFinal pruning comparison saved:")
print(comparison_output_path)

display(
    comparison_df[display_columns].sort_values(
        by=["mac_reduction_percent", "best_val_accuracy"],
        ascending=[False, False]
    )
)

Loaded: safe_structured_pruning_5_percent_summary.csv
Loaded: exploratory_pruning_6_percent_summary.csv
Loaded: exploratory_pruning_7_percent_summary.csv
Loaded: exploratory_pruning_8_percent_summary.csv
Loaded: exploratory_pruning_9_percent_summary.csv
Loaded: exploratory_pruning_10_percent_summary.csv
Loaded: exploratory_pruning_11_percent_summary.csv

Final pruning comparison saved:
/content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/results/pruning_5_6_7_8_9_10_11_final_comparison.csv


/tmp/ipykernel_1158/3346446922.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  comparison_df["saved_for_comparison"] = comparison_df["saved_for_comparison"].fillna(True)
/tmp/ipykernel_1158/3346446922.py:39: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  comparison_df["saved_for_comparison"] = comparison_df["saved_for_comparison"] | comparison_df["accepted_for_saving"].fillna(False)


,model_name,pruning_ratio,saved_for_comparison,candidate_quality_note,best_val_accuracy,best_val_macro_f1,accuracy_drop_percentage_point,f1_drop_percentage_point,parameter_reduction_percent,mac_reduction_percent,final_params_million,final_macs_million,saved_model_path
9,Exploratory Safe Structured Pruned MobileNetV2...,0.11,True,Possible trade-off candidate: moderate accurac...,0.986251,0.982698,1.068009,1.385383,20.569712,20.157554,1.805093,254.701367,/content/drive/MyDrive/Thesis/thesis_project/0...
7,Exploratory Safe Structured Pruned MobileNetV2...,0.10,True,Possible trade-off candidate: moderate accurac...,0.986005,0.980245,1.092561,1.630674,18.623661,18.730416,1.849318,259.254010,/content/drive/MyDrive/Thesis/thesis_project/0...
8,Exploratory Safe Structured Pruned MobileNetV2...,0.10,True,Possible trade-off candidate: moderate accurac...,0.986005,0.980245,1.092561,1.630674,18.623661,18.730416,1.849318,259.254010,/content/drive/MyDrive/Thesis/thesis_project/0...
6,Exploratory Safe Structured Pruned MobileNetV2 9%,0.09,True,Possible trade-off candidate: moderate accurac...,0.988583,0.983424,0.834766,1.312712,16.968604,16.859439,1.886930,265.222519,/content/drive/MyDrive/Thesis/thesis_project/0...
4,Exploratory Safe Structured Pruned MobileNetV2 8%,0.08,True,Strong candidate: accuracy and F1 drop within ...,0.991652,0.988267,0.527866,0.828391,15.192801,15.149395,1.927286,270.677644,/content/drive/MyDrive/Thesis/thesis_project/0...
5,Exploratory Safe Structured Pruned MobileNetV2 8%,0.08,True,Strong candidate: accuracy and F1 drop within ...,0.991652,0.988267,0.527866,0.828391,15.192801,15.149395,1.927286,270.677644,/content/drive/MyDrive/Thesis/thesis_project/0...
3,Exploratory Safe Structured Pruned MobileNetV2 7%,0.07,True,Strong candidate: accuracy and F1 drop within ...,0.990670,0.986826,0.626074,0.972574,13.541616,13.802790,1.964810,274.973380,/content/drive/MyDrive/Thesis/thesis_project/0...
1,Exploratory Safe Structured Pruned MobileNetV2 6%,0.06,True,Strong candidate: accuracy and F1 drop within ...,0.993125,0.990857,0.380555,0.569397,11.570130,11.336898,2.009613,282.839697,/content/drive/MyDrive/Thesis/thesis_project/0...
2,Exploratory Safe Structured Pruned MobileNetV2 6%,0.06,True,Strong candidate: accuracy and F1 drop within ...,0.993125,0.990857,0.380555,0.569397,11.570130,11.336898,2.009613,282.839697,/content/drive/MyDrive/Thesis/thesis_project/0...
0,Safe Structured Pruned MobileNetV2 5%,0.05,True,Strong candidate: accuracy and F1 drop within ...,0.993616,0.991108,0.331451,0.544307,9.637808,10.000355,2.053526,287.103338,/content/drive/MyDrive/Thesis/thesis_project/0...


In [45]:
# Cell 27C: Select the best pruned model from the final comparison table
# This cell selects the best pruning candidate using validation accuracy/F1 threshold and MAC reduction.

import shutil
import pandas as pd
from pathlib import Path

comparison_path = RESULTS_DIR / "pruning_5_6_7_8_9_10_11_final_comparison.csv"
comparison_df = pd.read_csv(comparison_path)

MAX_ACC_DROP_PP = 2.0
MAX_F1_DROP_PP = 2.0

eligible_df = comparison_df[
    (comparison_df["accuracy_drop_percentage_point"] <= MAX_ACC_DROP_PP) &
    (comparison_df["f1_drop_percentage_point"] <= MAX_F1_DROP_PP)
].copy()

if len(eligible_df) == 0:
    raise ValueError("No eligible pruning candidate found under the selected thresholds.")

best_candidate = eligible_df.sort_values(
    by=["mac_reduction_percent", "parameter_reduction_percent", "best_val_accuracy"],
    ascending=[False, False, False]
).iloc[0]

source_model_path = Path(best_candidate["saved_model_path"])
final_pruned_model_path = MODEL_DIR / "mobilenetv2_pruned_best_full_model.pth"

if not source_model_path.exists():
    raise FileNotFoundError(f"Selected source model not found: {source_model_path}")

shutil.copy2(source_model_path, final_pruned_model_path)

best_summary_path = RESULTS_DIR / "best_pruned_model_summary.csv"

best_df = pd.DataFrame([best_candidate])
best_df["selection_reason"] = (
    "The final pruned model was selected based on the highest MAC and parameter reduction "
    "while keeping validation accuracy and macro F1-score drop within the predefined acceptable threshold."
)
best_df["final_pruned_model_path"] = str(final_pruned_model_path)

best_df.to_csv(best_summary_path, index=False)

print("Selected best pruned model:")
display(best_df)

print("\nFinal selected pruned model copied to:")
print(final_pruned_model_path)

print("\nBest pruned model summary saved to:")
print(best_summary_path)

Selected best pruned model:


,model_name,pruning_ratio,saved_model_path,baseline_val_accuracy,baseline_val_macro_f1,baseline_params,baseline_macs,pre_finetune_val_accuracy,pre_finetune_val_macro_f1,best_val_accuracy,...,mac_reduction_percent,accuracy_drop_percentage_point,f1_drop_percentage_point,fine_tune_time_seconds,fine_tune_epochs,accepted_for_saving,saved_for_comparison,candidate_quality_note,selection_reason,final_pruned_model_path
9,Exploratory Safe Structured Pruned MobileNetV2...,0.11,/content/drive/MyDrive/Thesis/thesis_project/0...,0.996931,0.996551,2272550,319004966.0,0.012644,0.004506,0.986251,...,20.157554,1.068009,1.385383,643.148116,3,NaN,True,Possible trade-off candidate: moderate accurac...,The final pruned model was selected based on t...,/content/drive/MyDrive/Thesis/thesis_project/0...



Final selected pruned model copied to:
/content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/models/mobilenetv2_pruned_best_full_model.pth

Best pruned model summary saved to:
/content/drive/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/results/best_pruned_model_summary.csv
